# 08 — Hidden information margin (model 3) vs momentum + volatility

Two opposite defensive philosophies, on the same out-of-sample window:

- **momentum + volatility** (`momentum_volatility_rolling`) — prices the direction: it displays a
  P(up) curve calibrated per volatility regime, with the standard vig.
- **hidden margin (model 3)** (`hidden_symmetric_margin`) — hides the direction: it always shows
  P(up) = 0.50 and charges a symmetric information margin that widens when its hidden logistic
  signal says the move is predictable.

We compare them under uninformed noise flow and against the three informed attackers.

In [ ]:
import os
import sys

# Move up one level to the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# Add the new current directory to the Python path
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from snapmarket.data import load_oracle_prices, load_fast_feed
from snapmarket.features import build_features
from snapmarket.parameters import SharedParameters
from snapmarket.models import build_model
from snapmarket.experiments import common_evaluation_start
from snapmarket.signals import walk_forward_logistic_probability, regime_conditional_probability
from snapmarket.strategies import (
    noise_pool, predictive_bettor, lead_lag_bettor, regime_aware_bettor,
)
from snapmarket.engine import simulate

shared_parameters = SharedParameters()
features = build_features(load_oracle_prices(), shared_parameters)

models = {
    "momentum + volatility": build_model("momentum_volatility_rolling", features, shared_parameters),
    "hidden margin (model 3)": build_model("hidden_symmetric_margin", features, shared_parameters),
}
model_labels = list(models)
color_map = {model_labels[0]: "#9ca3af", model_labels[1]: "#2563eb"}

evaluation_start = common_evaluation_start(models.values())
print(f"common evaluation start: second {evaluation_start:,} (~{evaluation_start / 86_400:.0f} days in)")

## Part 1 — Uninformed random flow

Model 3 charges its information margin even on noise, so it should keep more vig (at higher
variance), while the momentum + volatility model sits near the base vig.

In [ ]:
window_length = 200_000
number_of_windows = 8

rows = []
for window_index in range(number_of_windows):
    start = evaluation_start + window_index * window_length
    if start + window_length + shared_parameters.horizon_seconds >= features.number_of_seconds:
        break
    for label, model in models.items():
        result = simulate(model, features, {"noise": noise_pool()}, start, window_length,
                          seed=1_000 + window_index)
        rows.append({"window": window_index, "model": label,
                     "house_edge": result.house_edge, "house_pnl": result.house_pnl,
                     "volume": result.total_volume})

results = pd.DataFrame(rows)
summary = (results.groupby("model")
           .agg(mean_house_edge=("house_edge", "mean"), std_house_edge=("house_edge", "std"),
                total_house_pnl=("house_pnl", "sum"), total_volume=("volume", "sum"))
           .reindex(model_labels))
summary["aggregate_house_edge"] = summary["total_house_pnl"] / summary["total_volume"]
print(summary.to_string(formatters={
    "mean_house_edge": "{:+.3%}".format, "std_house_edge": "{:.3%}".format,
    "total_house_pnl": "${:,.0f}".format, "total_volume": "${:,.0f}".format,
    "aggregate_house_edge": "{:+.3%}".format}))

## Part 2 — Informed attackers

Model 3 is built to defeat exactly the predictive (logistic) attacker, since it charges margin on
that very signal. The regime-aware attacker uses a different signal (the regime mispricing), which
model 3 does not directly price.

In [ ]:
fast_feed = load_fast_feed(expected_length=features.number_of_seconds)
logistic_probability = walk_forward_logistic_probability(features, shared_parameters)
regime_probability = regime_conditional_probability(features, shared_parameters)
informed_attackers = {
    "predictive (logistic)": predictive_bettor(logistic_probability),
    "lead-lag (fast feed)": lead_lag_bettor(features, fast_feed.log_price),
    "regime-aware": regime_aware_bettor(regime_probability),
}

informed_window_length = 150_000
informed_number_of_windows = 6

attacker_rows = []
for model_label, model in models.items():
    for attacker_label, bettor in informed_attackers.items():
        attacker_pnl = attacker_stake = total_volume = 0.0
        for window_index in range(informed_number_of_windows):
            start = evaluation_start + window_index * informed_window_length
            if start + informed_window_length + shared_parameters.horizon_seconds >= features.number_of_seconds:
                break
            result = simulate(model, features, {"pool": noise_pool(), "attacker": bettor},
                              start, informed_window_length, seed=200 + window_index)
            outcome = result.per_bettor["attacker"]
            attacker_pnl += outcome.pnl
            attacker_stake += outcome.stake
            total_volume += result.total_volume
        attacker_rows.append({"attacker": attacker_label, "model": model_label,
                              "attacker_edge": attacker_pnl / attacker_stake if attacker_stake else 0.0,
                              "volume_share": attacker_stake / total_volume if total_volume else 0.0})

attacker_results = pd.DataFrame(attacker_rows)
attacker_edge = attacker_results.pivot(index="attacker", columns="model", values="attacker_edge")[model_labels]
print(attacker_edge.to_string(formatters={label: "{:+.2%}".format for label in model_labels}))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
attacker_edge.plot(kind="bar", ax=ax, color=color_map)
ax.axhline(0, color="#16a34a", ls="--", label="break-even (beats the house above this)")
ax.axhline(-shared_parameters.house_margin, color="#dc2626", ls=":",
           label=f"uninformed baseline (-vig = {-shared_parameters.house_margin:.1%})")
ax.set_title("Informed attacker edge: momentum + volatility vs hidden margin (model 3)")
ax.set_ylabel("attacker edge"); ax.set_xlabel("")
ax.legend(fontsize=8)
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## Recorded results

Uninformed flow: 8 windows of 200,000 seconds. Informed attackers: 6 windows of 150,000 seconds.
Common out-of-sample start ~90 days in.

**Uninformed house edge**

| model | aggregate house edge | window standard deviation |
|---|---|---|
| momentum + volatility | +12.88% | 0.20% |
| hidden margin (model 3) | **+13.73%** | 0.49% |

**Informed attacker edge** (positive = beats the house)

| attacker | momentum + volatility | hidden margin (model 3) |
|---|---|---|
| predictive (logistic) | -4.04% | **-48.97%** |
| lead-lag (fast feed) | -17.18% | -18.03% |
| regime-aware | +2.97% | +3.54% |

**Read.** The two models defend against different attackers. Model 3 is purpose-built to beat the
predictive (logistic) attacker — it charges margin on that exact signal — so it destroys it
(-48.97%) and keeps more vig on noise (+13.73%), at higher variance. But it hides the direction
rather than pricing it, so it is slightly more exposed to the regime-aware attacker (+3.54% vs
+2.97%), whose edge comes from a signal model 3 does not price. Neither model dominates; the
natural next step is defence in depth: price the direction (momentum + volatility, or logistic)
**and** charge an information margin (model 3's mechanism) on the residual.

![attacker edge](assets/08_model3_vs_momentum_volatility_attackers.png)